# Parda — Phase 2: OCR benchmark (English / Hindi / Kannada)

**Settings:** Accelerator **GPU T4 x1** · Internet **On** · Input: add your dataset **parda-synth-v1**
Secrets: `GITHUB_TOKEN` (and `HF_TOKEN` only if you use the Hugging Face fallback in cell 3).

Runs 3 OCR engines on the same 300 pages (100 per language) and measures how much personal data each can read.
Total time ≈ 30–45 min. Best run as **Save Version → Save & Run All** after a quick check of cells 1–4.

In [ ]:
# 1. Get the latest code
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GH = secrets.get_secret("GITHUB_TOKEN")
REPO, CODE = "chetanjagan/Parda", "/tmp/Parda"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "-q", f"https://{GH}@github.com/{REPO}.git", CODE], check=True)
    subprocess.run(["git", "-C", CODE, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull", "-q", f"https://{GH}@github.com/{REPO}.git", "main"], check=True)
os.chdir(CODE)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install Tesseract (+hin, +kan), EasyOCR, rapidfuzz. Last lines must show kan/hin and a GPU name
!bash scripts/setup_ocr.sh

In [ ]:
# 3. Find the dataset. Works whether Kaggle extracted the zip or not (extracts to /tmp if needed).
#    Fallback: download from your Hugging Face backup (~5 min)
from parda.ocr.run_ocr import find_data, _n_images
try:
    DATA = find_data()
except FileNotFoundError:
    print("Kaggle dataset not attached -> downloading from Hugging Face backup")
    import glob, tarfile
    from huggingface_hub import snapshot_download
    p = snapshot_download("chetan-0804/parda-synth-v1", repo_type="dataset",
                          token=secrets.get_secret("HF_TOKEN"), local_dir="/tmp/hf_synth")
    os.makedirs("/tmp/parda_synth_v1", exist_ok=True)
    for t in sorted(glob.glob(f"{p}/images_*.tar")):
        tarfile.open(t).extractall("/tmp/parda_synth_v1")
    for f in glob.glob(f"{p}/*.json*"):
        os.replace(f, f"/tmp/parda_synth_v1/{os.path.basename(f)}")
    DATA = find_data()
n_ann = sum(1 for _ in open(f"{DATA}/annotations.jsonl", encoding="utf-8"))
print("DATA =", DATA)
print("images:", _n_images(DATA), "| annotation records:", n_ann, " <- both should be 30000")
OUT = "/kaggle/working/ocr_bench"


In [ ]:
# 4. Tests (~1-2 min): every line should say ok
!python tests/test_synth.py && python tests/test_ocr.py

In [ ]:
# 5. Tesseract (CPU, 4 processes) ~5-10 min
!python -m parda.ocr.run_ocr --engine tesseract --data {DATA} --out {OUT} --per_lang 100 --workers 4

In [ ]:
# 6. EasyOCR (GPU) ~10-15 min, first run downloads its models
!python -m parda.ocr.run_ocr --engine easyocr --data {DATA} --out {OUT} --per_lang 100

In [ ]:
# 7. OPTIONAL PaddleOCR (GPU). Paddle's GPU install is fussy on Kaggle; if this cell errors,
#    skip it — the benchmark works with the other two engines.
RUN_PADDLE = False
if RUN_PADDLE:
    !pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
    !pip install -q paddleocr
    !python -m parda.ocr.run_ocr --engine paddleocr --data {DATA} --out {OUT} --per_lang 100

In [ ]:
# 8. Score everything and show the report
!python -m parda.ocr.evaluate --out {OUT} --data {DATA} --per_lang 100 > /dev/null
from IPython.display import Markdown, Image as IPImage, display
display(Markdown(open(f"{OUT}/report.md").read()))
display(IPImage(f"{OUT}/chart_by_language.png"))
display(IPImage(f"{OUT}/chart_by_entity.png"))

In [ ]:
# 9. Error analysis: the worst PII misreads (what OCR saw vs the truth)
import json, itertools
for line in itertools.islice(open(f"{OUT}/failures.jsonl", encoding="utf-8"), 25):
    x = json.loads(line)
    print(f"{x['engine']:10s} {x['lang']:6s} {x['label']:13s} GT={x['gt']!r:40s} OCR={x['ocr'][:70]!r}")

In [ ]:
# 10. Pack the results to commit into GitHub (download from the Output tab)
!mkdir -p /kaggle/working/phase2_results && cp {OUT}/report.md {OUT}/report.json {OUT}/*.png /kaggle/working/phase2_results/
!ls -la /kaggle/working/phase2_results